# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tracy030115/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Flag a page as a Refresh Candidate if it hasn't been meaningfully updated in a long time, and its click-through rate is lower than you'd expect given how well it's already ranking. The rule can output reason codes depending on what it finds. It can output label STALE_LOW_CTR that marks the main refresh content, where a page is both stale and has a underperforming CTR. STALE_ONLY flags pages that are stale but whose CTR still looks normal. CTR_ONLY catches pages with weak CTR despite being recently updated.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    staleness AS (
        SELECT
            dc.content_hash_id,
            DATE_DIFF('day', dc.content_updated_date, ref.max_date) AS days_since_last_update
        FROM read_parquet('{rel}/dim_content.parquet') dc
        CROSS JOIN ref
    )
    SELECT
        CASE
            WHEN s.days_since_last_update <= 30 THEN '1_fresh_0-30d'
            WHEN s.days_since_last_update <= 180 THEN '2_moderate_31-180d'
            ELSE '3_stale_180d+'
        END AS staleness_bucket,
        COUNT(DISTINCT s.content_hash_id) AS n,
        AVG(f.gsc_clicks) AS avg_clicks,
        AVG(f.gsc_impressions) AS avg_impressions
    FROM staleness s
    JOIN read_parquet('{rel}/fact_content_daily_performance_sample.parquet') f
        ON s.content_hash_id = f.content_hash_id
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_clicks,avg_impressions
0,1_fresh_0-30d,142871,0.235757,37.176250
1,2_moderate_31-180d,258913,0.036821,9.195252
2,3_stale_180d+,7421,0.001768,0.425479


In [9]:
con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1_pos_1-3'
            WHEN gsc_avg_position <= 10 THEN '2_pos_4-10'
            WHEN gsc_avg_position <= 20 THEN '3_pos_11-20'
            ELSE '4_pos_21plus'
        END AS position_bucket,
        COUNT(*) AS n,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM (
        SELECT gsc_avg_position, gsc_clicks, gsc_impressions
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        WHERE gsc_impressions > 0
    )
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,total_clicks,total_impressions,ctr
0,1_pos_1-3,440501,387249.0,13927471.0,0.027805
1,2_pos_4-10,1530594,657613.0,155493934.0,0.004229
2,3_pos_11-20,633342,101416.0,24236904.0,0.004184
3,4_pos_21plus,1274500,62839.0,22536563.0,0.002788


Staleness: CONFIRMED

CTR-vs-position: MIXED

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

page_level = con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    fact_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            BOOL_OR(gsc_data_available) AS gsc_data_available_any
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    staleness AS (
        SELECT
            dc.content_hash_id,
            DATE_DIFF('day', dc.content_updated_date, ref.max_date) AS days_since_last_update
        FROM read_parquet('{rel}/dim_content.parquet') dc
        CROSS JOIN ref
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.total_impressions,
        f.total_clicks,
        f.avg_position,
        f.gsc_data_available_any,
        s.days_since_last_update,
        CASE WHEN f.total_impressions > 0
             THEN f.total_clicks * 1.0 / f.total_impressions
             ELSE NULL
        END AS ctr
    FROM fact_agg f
    LEFT JOIN staleness s ON f.content_hash_id = s.content_hash_id
""").df()

print(page_level.shape)
page_level.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(409205, 8)


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,gsc_data_available_any,days_since_last_update,ctr
0,client_3ffa76342f366962,content_1a6296faee432dae,0.0,0.0,NaN,False,41,NaN
1,client_3ffa76342f366962,content_dc34c661d63e55a9,0.0,0.0,NaN,False,41,NaN
2,client_3ffa76342f366962,content_dd83cb75985afc9c,3.0,0.0,6.666667,True,41,0.0
3,client_3ffa76342f366962,content_42e4dc3c4026a190,0.0,0.0,NaN,False,41,NaN
4,client_3ffa76342f366962,content_d86e1c84849226ba,0.0,0.0,NaN,False,41,NaN


In [15]:
def position_bucket(pos):
    if pos is None or pd.isna(pos):
        return None
    if pos <= 3:
        return "1_pos_1-3"
    elif pos <= 10:
        return "2_pos_4-10"
    elif pos <= 20:
        return "3_pos_11-20"
    else:
        return "4_pos_21plus"

# expected CTR per bucket, taken directly from the verified bucket table
expected_ctr_by_bucket = {
    "1_pos_1-3": 0.027805,
    "2_pos_4-10": 0.004229,
    "3_pos_11-20": 0.004184,
    "4_pos_21plus": 0.002788,
}

page_level["position_bucket"] = page_level["avg_position"].apply(position_bucket)
page_level["expected_ctr"] = page_level["position_bucket"].map(expected_ctr_by_bucket)

# positive gap = underperforming relative to its bucket's expected CTR
page_level["ctr_gap"] = page_level["expected_ctr"] - page_level["ctr"]

In [18]:
MIN_IMPRESSIONS = 10
STALE_THRESHOLD_DAYS = 180

def assign_reason_code(row):
    if (not row["gsc_data_available_any"]) or pd.isna(row["ctr"]) or row["total_impressions"] < MIN_IMPRESSIONS:
        return "INSUFFICIENT_DATA"

    is_stale = pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > STALE_THRESHOLD_DAYS
    is_low_ctr = pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0

    if is_stale and is_low_ctr:
        return "STALE_LOW_CTR"
    elif is_stale and not is_low_ctr:
        return "STALE_ONLY"
    elif not is_stale and is_low_ctr:
        return "CTR_ONLY"
    else:
        return "HEALTHY"

page_level["reason_code"] = page_level.apply(assign_reason_code, axis=1)
page_level["reason_code"].value_counts()

,count
reason_code,
INSUFFICIENT_DATA,247661
CTR_ONLY,109949
HEALTHY,51236
STALE_LOW_CTR,301
STALE_ONLY,58


In [19]:
STALE_CAP_DAYS = 730
page_level["staleness_norm"] = (
    page_level["days_since_last_update"].clip(lower=0, upper=STALE_CAP_DAYS) / STALE_CAP_DAYS
)

max_gap = page_level["ctr_gap"].clip(lower=0).quantile(0.99)
page_level["ctr_gap_norm"] = (page_level["ctr_gap"].clip(lower=0, upper=max_gap) / max_gap).fillna(0)

page_level["action_score"] = (
    0.5 * page_level["staleness_norm"].fillna(0) +
    0.5 * page_level["ctr_gap_norm"]
)

page_level.loc[page_level["reason_code"] == "INSUFFICIENT_DATA", "action_score"] = np.nan

In [20]:
KEEP_CODES = ["STALE_LOW_CTR", "STALE_ONLY", "CTR_ONLY"]

actionable = page_level[page_level["reason_code"].isin(KEEP_CODES)].copy()

ranked = actionable.sort_values("action_score", ascending=False, na_position="last").reset_index(drop=True)
ranked["rank"] = ranked.index + 1

output_cols = [
    "rank", "client_hash_id", "content_hash_id",
    "action_score", "reason_code",
    "days_since_last_update", "avg_position", "ctr", "expected_ctr", "ctr_gap",
    "total_impressions", "total_clicks", "gsc_data_available_any"
]

os.makedirs("work/outputs", exist_ok=True)
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
print(ranked["reason_code"].value_counts())
ranked[output_cols].head(10)

Wrote 110308 rows to work/outputs/baseline_action_score.csv
reason_code
CTR_ONLY         109949
STALE_LOW_CTR       301
STALE_ONLY           58
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,action_score,reason_code,days_since_last_update,avg_position,ctr,expected_ctr,ctr_gap,total_impressions,total_clicks,gsc_data_available_any
0,1,client_3ffa76342f366962,content_315f75aa07a662bf,0.658904,STALE_LOW_CTR,232,2.361667,0.0,0.027805,0.027805,48.0,0.0,True
1,2,client_3ffa76342f366962,content_84b05d1d2d5a9efd,0.658904,STALE_LOW_CTR,232,1.833333,0.0,0.027805,0.027805,10.0,0.0,True
2,3,client_a80fca3f171ed1de,content_973049f1eeccffa6,0.586986,CTR_ONLY,127,2.350866,0.0,0.027805,0.027805,40.0,0.0,True
3,4,client_a80fca3f171ed1de,content_7d385fd64452fd29,0.586986,CTR_ONLY,127,1.038987,0.0,0.027805,0.027805,473.0,0.0,True
4,5,client_20259bd6705d81d4,content_d105e79bf855829c,0.586986,CTR_ONLY,127,1.977937,0.0,0.027805,0.027805,132.0,0.0,True
5,6,client_23a62021009f63c4,content_12931963d6908b2f,0.585616,CTR_ONLY,125,1.400000,0.0,0.027805,0.027805,19.0,0.0,True
6,7,client_65de48885f4ef01b,content_f38f080cde16b0e5,0.585616,CTR_ONLY,125,0.142857,0.0,0.027805,0.027805,10.0,0.0,True
7,8,client_73cda7b4e4f265ea,content_41a048e44094be19,0.585616,CTR_ONLY,125,2.345833,0.0,0.027805,0.027805,42.0,0.0,True
8,9,client_73cda7b4e4f265ea,content_1c3cadd8a8560665,0.585616,CTR_ONLY,125,2.598485,0.0,0.027805,0.027805,20.0,0.0,True
9,10,client_73cda7b4e4f265ea,content_f99545159bcdc96c,0.585616,CTR_ONLY,125,2.871795,0.0,0.027805,0.027805,21.0,0.0,True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 1.
**Action:** Refresh — update content and metadata (title/meta description), since page is both stale (232 days) and earning zero clicks despite strong average position (2.36).

**Reason code:** STALE_LOW_CTR

**Confidence:** Low. Only 48 impressions and 0 clicks — a single click would move CTR from 0% to ~2%, right in line with the expected bucket rate. Zero clicks on low volume isn't strong evidence of a real problem.

**What would make it wrong:** If this keyword/page pairing is low-intent or informational (searchers get their answer from the snippet and don't need to click), 0% CTR could be normal for this query type, not a fixable page issue.

### 2.
**Action:** Refresh — same client, same staleness window (232 days), 0 clicks on strong position.

**Reason code:** STALE_LOW_CTR

**Confidence:** Very low. Only 10 impressions at the`MIN_IMPRESSIONS` cutoff, the minimum threshold for being considered "data available" at all. Barely above the line for trusting any CTR estimate here.

**What would make it wrong:** With only 10 impressions, this could easily be a page that simply hasn't accumulated enough traffic yet to reflect its true CTR, not a genuinely broken page.

### 3.
**Action:** CTR fix — check title tag/meta description/snippet, since staleness isn't the flag here (127 days, under the 180-day threshold) but CTR is 0% at position 2.35.

**Reason code:** CTR_ONLY

**Confidence:** Low-moderate. 40 impressions is more than the minimum, but still thin for confidently calling 0% CTR a real pattern rather than noise.

**What would make it wrong:** If this page recently changed rankings (jumped to position 2 very recently), the 0-click history might just reflect the old, worse position, which is not a CTR problem at the current position.

### 4.
**Action:** CTR fix — same client/window, but much higher impression volume.

**Reason code:** CTR_ONLY

**Confidence:** Higher than the rest of the top 10. 473 impressions with 0 clicks at position ~1 is a real, striking pattern — this is the strongest case in the batch that something (title/snippet) is actually broken.

**What would make it wrong:** If `gsc_clicks` tracking has a bug or gap for this specific content/client pair (e.g. GSC not properly linked, or clicks logged under a different URL variant due to redirects), the 0 could be a tracking artifact rather than true searcher behavior.

### 5.
**Action:** CTR fix — 132 impressions, 0 clicks, position ~2.

**Reason code:** CTR_ONLY

**Confidence:** Moderate. Reasonable impression volume for a CTR read, though still small relative to the traffic leaders in the full dataset.

**What would make it wrong:** If this page's actual displayed snippet differs from what's assumed (e.g. a rich result or featured snippet is capturing clicks without registering as a standard "click" in GSC), the fix wouldn't be a title/meta rewrite.

### 6.
**Action:** CTR fix — 125 days since update, 19 impressions, 0 clicks.

**Reason code:** CTR_ONLY

**Confidence:** Low. 19 impressions is thin; a single click changes the picture substantially.

**What would make it wrong:** Low absolute volume means this could just be a low-traffic long-tail page where 0 clicks over a short window is statistically unremarkable.

### 7.
**Action:** Flag for a data check before acting, not a content fix. `avg_position = 0.14` is below the valid range for a search ranking (position 1 is the best possible rank; a value under 1 isn't real).

**Reason code:** CTR_ONLY

**Confidence:** Untrustworthy as-is. This is very likely a data quality issue in the position calculation for this row, not a genuine CTR problem, so the flag itself can't be relied on yet.

**What would make it wrong:** If the position averaging pulled in a bad or placeholder value for this content/date combination, the bucket assignment and the resulting flag are both invalid until that's fixed.

### 8.
**Action:** CTR fix — 42 impressions, 0 clicks, position 2.35, 125 days since update.

**Reason code:** CTR_ONLY

**Confidence:** Low-moderate, similar to row 3.

**What would make it wrong:** Same caveat as others in this range, it could reflect a recent ranking change not yet reflected in accumulated click history.

### 9.
**Action:** CTR fix — same client as row 8, 20 impressions, 0 clicks, position 2.6.

**Reason code:** CTR_ONLY

**Confidence:** Low. Small impression count.

**What would make it wrong:** This client appears three times in the top 10 (rows 8, 9, 10). Worth checking whether this is a client-level tracking issue (e.g. a GSC connection reset, or a redirect/canonicalization change affecting multiple pages at once) rather than three independent content problems.

### 10.
**Action:** CTR fix — 21 impressions, 0 clicks, position 2.87.

**Reason code:** CTR_ONLY

**Confidence:** Low, and compounded by the same-client repetition noted in row 9.

**What would make it wrong:** If this is a client-wide GSC data gap rather than three separate content problems, the fix isn't three individual CTR rewrites — it's checking that client's tracking setup first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks — which ones look wrong and why

Rows 2 has only 10 impressions at the minimum for being considered scoreable. Oneclick would change its CTR from 0% to roughly the expected rate for its bucket. This is a case where the reason code is technically correct by the rule's own thresholds, but the confidence behind it is essentially zero.

Row 7 has avg_position = 0.14. GSC positions start at 1, so it's likely a data artifact. This pick shouldn't be trusted until the position computation is checked; right now it's ranked as if it were a good position, when it may not represent a real ranking.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

used_cols = ["client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
             "gsc_avg_position", "gsc_data_available", "content_updated_date", "report_date"]

fact_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()["column_name"].tolist()
dim_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()["column_name"].tolist()

print("Fact columns NOT used:", [c for c in fact_cols if c not in used_cols])
print("Dim columns NOT used:", [c for c in dim_cols if c not in used_cols])


Fact columns NOT used: ['client_has_gsc', 'client_has_ga4', 'ga4_data_available', 'gsc_sum_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Dim columns NOT used: ['keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.